# MCP with Memory

In this notebook you give an MCP agent a **memory**, so that it can remember earlier questions and answers between calls. The history is stored in Redis, and it is summarised when it grows too long.

## Learning Objectives

At the end of this notebook, you should be able to:

- Store and reload chat history in Redis so that an agent remembers earlier turns.
- Summarise old messages to keep the history short.
- Run slow memory work in the background so the user waits less.
- Combine MCP tool calls with persistent memory in a single agent function.

## Why Does the Agent Need Memory?

An LLM remembers nothing between calls: each request only knows the messages you send with it. To hold a conversation, your code must store the earlier messages and send them again every time.

Here the messages are stored in **Redis**, a fast in-memory database. The alternative is a plain Python list, which is simpler but disappears when the notebook restarts. Redis keeps the history between runs.

Sending the whole history every time has a cost: it keeps growing, and a model can only read a limited amount of text. So when the history gets long, the agent asks the LLM to **summarise** the older messages and keeps only the summary plus the last four messages.

```mermaid
flowchart TD
    A[New question] --> B[Load history from Redis]
    B --> C[Ask the LLM, calling MCP tools if needed]
    C --> D[Answer the user]
    D --> E[Save history to Redis in the background]
    E --> F{More than 10 messages?}
    F -->|Yes| G[Summarise older messages, keep the last 4]
    F -->|No| H[Done]
    G --> H
```

## Setup

The next cells load the libraries, name the model, set the URL of the MCP server, and define the `START_FRESH` flag. Set `START_FRESH` to `True` to clear the memory at the start of a session.

This notebook uses the tools in `mcp_servers/mcp_tools_server.py`, so start that server in a separate terminal before you run the agent:

```bash
uv run python mcp_servers/mcp_tools_server.py
```

In [1]:
import asyncio
import json
import sys

import ollama
import redis
from mcp import ClientSession
from mcp.client.sse import sse_client

In [2]:
LARGE_MODEL = "llama3.2:3b"

In [3]:
LOCAL_SERVER_URL = "http://127.0.0.1:54321/sse"

In [4]:
START_FRESH = False  # Set to True to clear memory at the start of the session

Connect to Redis. Make sure your Redis server is running: `redis-cli ping` should answer `PONG`. The client uses `decode_responses=True`, so values come back as text instead of bytes.

In [5]:
# We use decode_responses=True to handle strings easily
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)

## Memory Helpers

### Listing and Clearing Sessions

The code block below defines two utility functions for managing chat history sessions stored in Redis.

1. `list_active_sessions()`
    - **Purpose**: Lists all active chat history sessions by querying Redis for keys matching the pattern `"chat_history:*"`.
    - **Behaviour**: Retrieves the keys using `r.keys()`, then prints them as a comma-separated list if any exist. Otherwise it prints "Cache is empty."
    - **Usage**: Call this function to inspect which memory keys are stored in Redis.

2. `clear_session(session_id="user_123")`
    - **Purpose**: Clears the chat history for a specific session by deleting its Redis key.
    - **Behaviour**: Builds the memory key as `f"chat_history:{session_id}"`, deletes it using `r.delete()`, and prints a confirmation message.
    - **Usage**: Call this function with a session ID (the default is `"user_123"`) to wipe that session's memory and start fresh.

In [6]:
def list_active_sessions():
    """Run this to see what memory keys exist in Redis."""
    keys = r.keys("chat_history:*")
    print(f"Active Sessions: {', '.join(keys)}" if keys else "Cache is empty.")


def clear_session(session_id="user_123"):
    """Run this to wipe a specific project and start over."""
    memory_key = f"chat_history:{session_id}"
    r.delete(memory_key)
    print(f"🧹 Wiped memory for: {session_id}")

### Summarising Long Histories

The function `summarize_and_save_memory` keeps the conversation history short. It summarises the history when it becomes too long, then saves the updated history to Redis so that it persists.

**Key steps:**

1. **Check the history length.** If there are more than 10 messages, it triggers summarisation.
2. **Summarise.**
    - Takes everything except the last 4 messages (the older context) for summarisation.
    - Keeps the last 4 messages (the recent context) unchanged.
    - Asks the LLM to summarise the older messages.
    - Creates a new "system" message containing the summary to replace the older context.
    - Combines the summary message with the recent messages to form the new history.
3. **Save to Redis.** Converts the messages to a serialisable format (using `.model_dump()` if available) and stores the JSON history in Redis under the given `memory_key`.

**Purpose:**

- Prevents memory bloat by compressing early parts of the conversation.
- Keeps storage and retrieval efficient across sessions.
- Logs progress to stderr for debugging without interrupting the user's output.

In [7]:
async def summarize_and_save_memory(messages, model, memory_key):
    # Only summarize if history is getting too long (e.g., > 10 messages)
    if len(messages) > 10:
        # print("🧠 History too long. Summarizing early messages...")
        sys.stderr.write("🧠 History too long...\n")

        # Summarize everything except the most recent 4 messages
        to_summarize = messages[:-4]
        # Keep the recent 4 messages exactly as they are
        recent_messages = messages[-4:]

        summary_prompt = {
            "role": "user",
            "content": f"Summarize the following conversation key points concisely: {json.dumps(to_summarize)}",
        }

        summary_resp = ollama.chat(model=model, messages=[summary_prompt])

        # Create a new 'Summary' message to act as the new foundation
        summary_message = {
            "role": "system",
            "content": f"Previously in this conversation: {summary_resp['message']['content']}",
        }

        # New history = [The Summary] + [The Recent Messages]
        messages = [summary_message] + recent_messages
        print(f"📉 Memory compressed. New length: {len(messages)}")

    # Save to Redis
    # Ensure everything is a dict using .model_dump() to avoid the TypeError
    serializable = [m.model_dump() if hasattr(m, "model_dump") else m for m in messages]
    r.set(memory_key, json.dumps(serializable))
    return messages

### Saving in the Background

The function `background_memory_tasks` schedules saving and summarisation as an asyncio task. The synchronous Ollama call used for summarisation can still block the notebook's event loop.

**Key steps:**

1. **Prepare a serialisable list.** Converts each message to a dictionary using `.model_dump()` if the method exists, so the list can be turned into JSON.
2. **Save to Redis.** Stores the messages as a JSON string under the given `memory_key`.
3. **Summarise if needed.** Calls `summarize_and_save_memory` to compress the history if it is too long. This is the slowest part, because it needs another LLM call.
4. **Log.** Writes a success or error message to stderr so it does not interrupt the user's view.

**Purpose:**

- Persists memory without freezing the application.
- Handles exceptions by logging errors for debugging.

In [8]:
async def background_memory_tasks(messages, model, memory_key):
    """Handles Redis saving and summarization in the background."""
    try:
        # 1. Prepare serializable list
        serializable = [
            m.model_dump() if hasattr(m, "model_dump") else m for m in messages
        ]

        # 2. Save to Redis
        r.set(memory_key, json.dumps(serializable))

        # 3. Summarize if needed (This is the slowest part!)
        await summarize_and_save_memory(messages, model, memory_key)

        # Log to stderr so it doesn't interrupt the user's view
        import sys

        sys.stderr.write("✅ Background: Memory synced and summarized.\n")
    except Exception as e:
        import sys

        sys.stderr.write(f"❌ Background Task Error: {e}\n")

## The System Prompt

The system prompt gives the model its rules: answer questions about the conversation from memory, use `search_web` only for facts from outside the conversation, and use `read_local_file` for files. Keep in mind that these rules are instructions the model may or may not follow, not guarantees. You will see an example in the results below.

In [9]:
system_prompt = (
    "You are a smart assistant with perfect memory of the conversations. "
    "Rule 1: If I ask you about something we already discussed, "
    "read the conversation history and answer directly. DO NOT use tools for this. "
    "Rule 2: CRITICAL RULE: You are strictly FORBIDDEN from using the 'search_web' tool "
    "if the user's prompt asks about 'I', 'you', 'we', 'history', 'just asked', or 'said'. "
    "If the answer is in your memory, reply immediately without tools. "
    "Only use 'search_web' for real-world facts outside of this conversation. "
    "Rule 3: If the conversation history gets too long, summarize the early parts to keep it concise. "
    "Rule 4: Use the 'read_local_file' tool if I ask you to read a file. "
    "Rule 5: ALWAYS save important facts to memory after you find them, so you can recall them later without searching again."
)

## The Memory-Aware Agent

The function `run_mcp_query_with_memory` combines everything. For each question it:

1. **Manages memory.** Loads the history for the session from Redis, or starts a new one if `START_FRESH` is `True` or nothing is stored yet, and adds the new question.
2. **Prepares the tools.** Lists the tools offered by the MCP server.
3. **Makes the first LLM call**, with the full history and the tools.
4. **Handles tool calls.** Runs each tool the model asks for, adds the results to the history, and streams the final answer.
5. **Syncs memory.** Starts `background_memory_tasks` as a background task, so saving and summarising do not delay the answer.

In [10]:
async def run_mcp_query_with_memory(user_prompt, system_prompt, model, session_id=1):
    async with sse_client(LOCAL_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            memory_key = f"chat_history:{session_id}"

            # --- 1. SILENT MEMORY MANAGEMENT ---
            # We handle Redis directly before the LLM even sees it
            if START_FRESH:
                print(f"🧹 START_FRESH is True. Wiping Redis key: {memory_key}")
                r.delete(memory_key)
                messages = [{"role": "system", "content": system_prompt}]
            else:
                existing_history = r.get(memory_key)
                if existing_history:
                    messages = json.loads(existing_history)
                    print(f"🧠 Memory Loaded: {len(messages)} messages found.")
                else:
                    messages = [{"role": "system", "content": system_prompt}]

            # Add the new user prompt to history
            messages.append({"role": "user", "content": user_prompt})

            # --- 2. PREPARE SERVER TOOLS ---
            # Because we cleaned the server, this ONLY fetches search_web and read_local_file
            tools = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.input_schema,
                    },
                }
                for t in tools.tools
            ]

            # --- 3. FIRST LLM CALL ---
            response = ollama.chat(
                model=model,
                messages=messages,
                tools=ollama_tools,
                options={"num_ctx": 8192, "temperature": 0},
            )

            # Convert the message to a plain dictionary and save to history
            assistant_msg = response["message"]
            if hasattr(assistant_msg, "model_dump"):
                assistant_msg = assistant_msg.model_dump()
            messages.append(assistant_msg)
            # --- 4. HANDLE TOOL CALLS (AGENTIC LOOP) ---
            if assistant_msg.get("tool_calls"):
                for tool_call in assistant_msg["tool_calls"]:
                    t_name = tool_call["function"]["name"]
                    t_args = tool_call["function"]["arguments"]

                    print(f"🛠️ Tool Call: {t_name} | Searching for: {t_args}")
                    result = await session.call_tool(t_name, t_args)

                    # Add tool result to history so the LLM knows it succeeded
                    messages.append(
                        {"role": "tool", "content": str(result.content), "name": t_name}
                    )

                # Get final thought from LLM after seeing tool results
                print("🤖 Assistant: ", end="", flush=True)
                final_content = ""

                # STREAMING THE RESPONSE
                stream = ollama.chat(model=model, messages=messages, stream=True)
                for chunk in stream:
                    content = chunk["message"]["content"]
                    print(content, end="", flush=True)
                    final_content += content

                print()  # Print new line when stream finishes

                # CRITICAL FIX: Save the final generated answer to the history list!
                messages.append({"role": "assistant", "content": final_content})

            else:
                # If no tools were called, the final content is just the first response
                final_content = assistant_msg["content"]
                print(f"🤖 Assistant: {final_content}")

            # --- 5. BACKGROUND MEMORY SYNC ---
            # Save the updated history (including tool results and final answer) without freezing the app
            asyncio.create_task(background_memory_tasks(messages, model, memory_key))

            return final_content

## Trying It Out

Run the questions below in order. Each call loads the history saved by the previous one.

In [11]:
query = "What is the capital of Germany?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

🛠️ Tool Call: search_web | Searching for: {'query': 'capital of Germany'}
🤖 Assistant: The capital of Germany is Berlin.


✅ Background: Memory synced and summarized.


'The capital of Germany is Berlin.'

In [12]:
query = "What did I just ask?"
await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL)

🧠 Memory Loaded: 5 messages found.
🛠️ Tool Call: search_web | Searching for: {'query': 'What is the capital of Germany?'}
🤖 Assistant: You just asked me what the capital of Germany is. I already answered that the capital of Germany is Berlin.


✅ Background: Memory synced and summarized.


'You just asked me what the capital of Germany is. I already answered that the capital of Germany is Berlin.'

In [13]:
query = "who is the actor in the movie Inception ?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 9 messages found.
🛠️ Tool Call: search_web | Searching for: {'query': 'actor in movie Inception'}
🤖 Assistant: You're referring to the movie Inception. The main actor in the movie Inception is Leonardo DiCaprio, who played the role of Cobb, a skilled thief who specializes in entering people's dreams.


🧠 History too long...


📉 Memory compressed. New length: 5
You're referring to the movie Inception. The main actor in the movie Inception is Leonardo DiCaprio, who played the role of Cobb, a skilled thief who specializes in entering people's dreams.


✅ Background: Memory synced and summarized.


In [14]:
query = "Read and summarize the contents in file ai_response.txt in the root folder"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 5 messages found.
🛠️ Tool Call: read_local_file | Searching for: {'file_path': 'ai_response.txt'}
🤖 Assistant: The contents of the file `ai_response.txt` are a text explaining the role of an AI Project Manager, comparing it to building a Lego castle. The text highlights the key responsibilities of an AI Project Manager, including planning, coordination, and quality control, and how they use tools and assistants to build successful projects.
The contents of the file `ai_response.txt` are a text explaining the role of an AI Project Manager, comparing it to building a Lego castle. The text highlights the key responsibilities of an AI Project Manager, including planning, coordination, and quality control, and how they use tools and assistants to build successful projects.


✅ Background: Memory synced and summarized.


In [15]:
query = "actor in the movie Inception?"
print(await run_mcp_query_with_memory(query, system_prompt, LARGE_MODEL))

🧠 Memory Loaded: 9 messages found.
🛠️ Tool Call: search_web | Searching for: {'query': 'actor in movie Inception'}
🤖 Assistant: The actor in the movie Inception is Leonardo DiCaprio, who played the role of Dom Cobb, a professional thief who specializes in conning secrets from his victims by infiltrating their dreams.


🧠 History too long...


📉 Memory compressed. New length: 5
The actor in the movie Inception is Leonardo DiCaprio, who played the role of Dom Cobb, a professional thief who specializes in conning secrets from his victims by infiltrating their dreams.


✅ Background: Memory synced and summarized.


### Reading the Results

These observations come from one run. Each `Memory Loaded` line in the output shows how many messages were read from Redis.

| Question | Memory loaded | Tool called | What happened |
|---|---|---|---|
| What is the capital of Germany? | none (new session) | `search_web` | Answered "The capital of Germany is Berlin." |
| What did I just ask? | 5 messages | `search_web` | Answered correctly from memory: you asked for the capital of Germany. |
| Who is the actor in the movie Inception? | 9 messages | `search_web` | Answered Leonardo DiCaprio as Dom Cobb. The history then exceeded 10 messages, so it was compressed to 5. |
| Read and summarise the contents of `ai_response.txt` | 5 messages | `read_local_file` | Summarised the Lego castle explanation of an AI Project Manager. |
| Actor in the movie Inception? | 9 messages | `search_web` | Listed several cast members, then compressed the history to 5 again. |

What to notice:

- **Memory works.** The second question only makes sense because the first exchange was loaded from Redis.
- **Prompt rules are advice, not guarantees.** The system prompt forbids `search_web` for questions about the conversation, yet the model still called it for "What did I just ask?". It answered correctly anyway because the history was part of its input.
- **The message counts follow the code.** The history starts with the system message. A question that needs a tool adds four more messages: the question, the model's tool request, the tool result, and the final answer. That gives 1 + 4 = 5, then 5 + 4 = 9, and 9 + 4 = 13. Thirteen is more than 10, so the older messages are summarised and the history shrinks to the summary plus the last 4 messages.
- **Summaries can lose detail.** The same small model writes the summary, so facts from the older messages may be shortened or dropped.

## Wipe the Memory for a Fresh Start

If the stored history gets confusing, delete it. The key must match the `session_id` you are using (the default here is `1`).

In [16]:
# Wipe the corrupted memory
r.delete("chat_history:1")  # Make sure the session ID matches what you are using
print("Memory wiped! Ready for a fresh start.")

Memory wiped! Ready for a fresh start.


## Summary

In this notebook you:

- Stored and reloaded chat history in Redis so that the agent remembered earlier turns.
- Summarised old messages to keep the history down to a summary plus the last four messages.
- Ran saving and summarising in a background task, so the user did not wait for them.
- Combined MCP tool calls with persistent memory in one agent function, and saw that the model does not always follow the rules in its prompt.

## References & Further Reading

- [**Redis Documentation**](https://redis.io/docs/latest/): Reference for the store behind the chat memory
- [**redis-py**](https://redis-py.readthedocs.io/en/stable/): The Python client used in this notebook
- [**asyncio**](https://docs.python.org/3/library/asyncio.html): Python's library for the asynchronous code used for background tasks
- [**Model Context Protocol**](https://modelcontextprotocol.io/): Official introduction and specification of MCP
- [**Ollama Tool Calling**](https://docs.ollama.com/capabilities/tool-calling): How a local model chooses and calls tools